# YOLO Dataset Preparation

This notebook converts the HuggingFace parking space dataset (XML annotations) into YOLO format for object detection training.

Steps:
1. Parse XML annotations
2. Convert bounding boxes to YOLO format (normalized coordinates)
3. Split into train/val sets
4. Create YOLO directory structure
5. Generate data.yaml configuration file


## 1. Imports and Path Setup


In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# HuggingFace dataset paths
# Note: annotations.xml is at hf_parking_space/annotations.xml (not inside data/)
HF_ROOT = BASE_DIR / "data" / "raw" / "hf_parking_space"
HF_IMAGES_DIR = HF_ROOT / "data" / "images"
HF_ANNOT_XML = HF_ROOT / "annotations.xml"  # XML is at root level, not in data/

# YOLO dataset output paths
YOLO_ROOT = BASE_DIR / "data" / "processed" / "yolo_parking"
IMG_TRAIN = YOLO_ROOT / "images" / "train"
IMG_VAL = YOLO_ROOT / "images" / "val"
LBL_TRAIN = YOLO_ROOT / "labels" / "train"
LBL_VAL = YOLO_ROOT / "labels" / "val"

# Create directories
for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("HF_ROOT:", HF_ROOT)
print("HF_IMAGES_DIR:", HF_IMAGES_DIR, "exists:", HF_IMAGES_DIR.exists())
print("HF_ANNOT_XML:", HF_ANNOT_XML, "exists:", HF_ANNOT_XML.exists())
print("YOLO_ROOT:", YOLO_ROOT)


BASE_DIR: c:\Harosha\George Brown\DL2\parking-vision
HF_ROOT: c:\Harosha\George Brown\DL2\parking-vision\data\raw\hf_parking_space
HF_IMAGES_DIR: c:\Harosha\George Brown\DL2\parking-vision\data\raw\hf_parking_space\data\images exists: True
HF_ANNOT_XML: c:\Harosha\George Brown\DL2\parking-vision\data\raw\hf_parking_space\annotations.xml exists: True
YOLO_ROOT: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking


## 2. Inspect XML Structure

In [2]:
# Inspect the XML structure to understand the format
if HF_ANNOT_XML.exists():
    tree = ET.parse(HF_ANNOT_XML)
    root = tree.getroot()
    
    print("Root tag:", root.tag)
    print("\nFirst 3 image elements:")
    for i, child in enumerate(list(root)[:3]):
        print(f"\n--- Image {i+1} ---")
        ET.dump(child)
        if i >= 2:
            break
else:
    print("XML file not found. Please check the path.")


Root tag: annotations

First 3 image elements:

--- Image 1 ---
<version>1.1</version>
  

--- Image 2 ---
<meta>
    <task>
      <segments>
        <segment>
          <id>32599</id>
          <start>0</start>
          <stop>32</stop>
          <url>https://cvat2.trainingdata.solutions/api/jobs/32599</url>
        </segment>
      </segments>
      <owner>
        <username>TrainingData</username>
        <email />
      </owner>
      
      <labels>
        <label>
          <name>free_parking_space</name>
          <color>#3d3df5</color>
          <type>polygon</type>
          <attributes>
            <attribute>
              <name>not_visible</name>
              <mutable>False</mutable>
              <input_type>checkbox</input_type>
              <default_value>false</default_value>
              <values>false</values>
            </attribute>
          </attributes>
        </label>
        <label>
          <name>not_free_parking_space</name>
          <color>#ff6037</colo

## 3. Parse XML Annotations


In [3]:
def parse_hf_xml(xml_path: Path):
    """
    Parse HuggingFace XML annotations (CVAT format).
    Returns a dictionary mapping image names to their annotations.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    images_info = {}  # {img_name: {"width": w, "height": h, "boxes": [dicts...] } }

    for img_elem in root.findall("image"):
        img_name = img_elem.get("name")
        img_w = float(img_elem.get("width"))
        img_h = float(img_elem.get("height"))

        boxes = []
        
        # Try different possible tag names for bounding boxes
        # CVAT uses "box", but some formats use "polygon" which we convert to bbox
        for box in img_elem.findall("box"):
            label = box.get("label")
            xtl = float(box.get("xtl"))
            ytl = float(box.get("ytl"))
            xbr = float(box.get("xbr"))
            ybr = float(box.get("ybr"))

            boxes.append({
                "label": label,
                "xtl": xtl,
                "ytl": ytl,
                "xbr": xbr,
                "ybr": ybr,
            })
        
        # Also handle polygon annotations (convert to bounding box)
        for polygon in img_elem.findall("polygon"):
            label = polygon.get("label")
            points_str = polygon.get("points")
            
            # Parse points: "x1,y1;x2,y2;x3,y3;x4,y4"
            points = []
            for point_pair in points_str.split(";"):
                if point_pair.strip():
                    x, y = map(float, point_pair.split(","))
                    points.append((x, y))
            
            if len(points) < 2:
                continue
            
            # Convert polygon to bounding box
            x_coords = [p[0] for p in points]
            y_coords = [p[1] for p in points]
            xtl = min(x_coords)
            ytl = min(y_coords)
            xbr = max(x_coords)
            ybr = max(y_coords)
            
            boxes.append({
                "label": label,
                "xtl": xtl,
                "ytl": ytl,
                "xbr": xbr,
                "ybr": ybr,
            })

        if boxes:  # Only add images that have annotations
            images_info[img_name] = {
                "width": img_w,
                "height": img_h,
                "boxes": boxes,
            }

    return images_info

# Parse the XML
images_info = parse_hf_xml(HF_ANNOT_XML)
print(f"Parsed {len(images_info)} images with annotations")
print(f"\nSample image names:")
for i, img_name in enumerate(list(images_info.keys())[:5]):
    img_info = images_info[img_name]
    print(f"  {i+1}. {img_name}: {len(img_info['boxes'])} boxes, size: {img_info['width']}x{img_info['height']}")


Parsed 30 images with annotations

Sample image names:
  1. images/0.png: 28 boxes, size: 1200.0x621.0
  2. images/1.png: 38 boxes, size: 650.0x487.0
  3. images/10.png: 20 boxes, size: 2560.0x1820.0
  4. images/11.png: 36 boxes, size: 1353.0x1041.0
  5. images/12.png: 39 boxes, size: 1920.0x1080.0


In [4]:
# Map labels to YOLO class IDs
label_to_id = {
    "free_parking_space": 0,
    "not_free_parking_space": 1,
    "partially_free_parking_space": 2,
}

id_to_label = {v: k for k, v in label_to_id.items()}

print("Label to ID mapping:")
for label, cls_id in label_to_id.items():
    print(f"  {cls_id}: {label}")

# Check which labels appear in the dataset
all_labels = set()
for img_info in images_info.values():
    for box in img_info["boxes"]:
        all_labels.add(box["label"])

print(f"\nLabels found in dataset: {sorted(all_labels)}")
print(f"Labels in mapping: {set(label_to_id.keys())}")

missing_labels = all_labels - set(label_to_id.keys())
if missing_labels:
    print(f"\nWARNING: Some labels are not in the mapping: {missing_labels}")
    print("These will be skipped during conversion.")


Label to ID mapping:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space

Labels found in dataset: ['free_parking_space', 'not_free_parking_space', 'partially_free_parking_space']
Labels in mapping: {'not_free_parking_space', 'partially_free_parking_space', 'free_parking_space'}


## 5. Train/Validation Split


In [5]:
# Split images into train and validation sets
all_img_names = list(images_info.keys())
train_imgs, val_imgs = train_test_split(
    all_img_names,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print(f"Total images: {len(all_img_names)}")
print(f"Train images: {len(train_imgs)} ({len(train_imgs)/len(all_img_names)*100:.1f}%)")
print(f"Val images: {len(val_imgs)} ({len(val_imgs)/len(all_img_names)*100:.1f}%)")


Total images: 30
Train images: 24 (80.0%)
Val images: 6 (20.0%)


## 6. Convert to YOLO Format


In [6]:
def boxes_to_yolo_lines(img_info, label_map):
    """
    Convert bounding boxes to YOLO format lines.
    YOLO format: class_id x_center y_center width height (all normalized to [0,1])
    """
    w = img_info["width"]
    h = img_info["height"]
    lines = []

    for b in img_info["boxes"]:
        label = b["label"]
        if label not in label_map:
            continue
        cls_id = label_map[label]

        x_min, y_min = b["xtl"], b["ytl"]
        x_max, y_max = b["xbr"], b["ybr"]

        # Calculate center and dimensions
        x_c = (x_min + x_max) / 2.0
        y_c = (y_min + y_max) / 2.0
        bw = (x_max - x_min)
        bh = (y_max - y_min)

        # Normalize to [0, 1]
        x_c_n = x_c / w
        y_c_n = y_c / h
        bw_n = bw / w
        bh_n = bh / h

        # Clip to valid range [0, 1]
        x_c_n = max(0.0, min(1.0, x_c_n))
        y_c_n = max(0.0, min(1.0, y_c_n))
        bw_n = max(0.0, min(1.0, bw_n))
        bh_n = max(0.0, min(1.0, bh_n))

        lines.append(f"{cls_id} {x_c_n:.6f} {y_c_n:.6f} {bw_n:.6f} {bh_n:.6f}")

    return lines

# Test the conversion on one image
if images_info:
    test_img_name = list(images_info.keys())[0]
    test_img_info = images_info[test_img_name]
    test_lines = boxes_to_yolo_lines(test_img_info, label_to_id)
    print(f"Test conversion for: {test_img_name}")
    print(f"  Original boxes: {len(test_img_info['boxes'])}")
    print(f"  YOLO lines: {len(test_lines)}")
    if test_lines:
        print(f"  Sample line: {test_lines[0]}")


Test conversion for: images/0.png
  Original boxes: 28
  YOLO lines: 28
  Sample line: 0 0.081633 0.171739 0.071933 0.319002


In [7]:
def process_split(img_names, img_dst_dir, lbl_dst_dir, images_info, label_map, split_name="train"):
    """
    Process a split (train or val): copy images and create YOLO label files.
    """
    count = 0
    skipped_no_image = 0
    skipped_no_labels = 0
    
    for img_name in tqdm(img_names, desc=f"Processing {split_name}"):
        if img_name not in images_info:
            continue

        img_info = images_info[img_name]

        # Source image path
        # Handle "images/0.png" format - extract just the filename
        img_filename = Path(img_name).name
        src_img_path = HF_IMAGES_DIR / img_filename
        
        if not src_img_path.exists():
            skipped_no_image += 1
            continue

        # Compute YOLO label lines
        lines = boxes_to_yolo_lines(img_info, label_map)
        if not lines:
            skipped_no_labels += 1
            continue

        # Copy image
        dst_img_path = img_dst_dir / img_filename
        shutil.copy2(src_img_path, dst_img_path)

        # Write label file (same name as image but with .txt extension)
        dst_lbl_path = lbl_dst_dir / (Path(img_filename).stem + ".txt")
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(lines))

        count += 1

    return count, skipped_no_image, skipped_no_labels

# Process train and validation splits
print("Processing train split...")
n_train, skipped_train_img, skipped_train_lbl = process_split(
    train_imgs, IMG_TRAIN, LBL_TRAIN, images_info, label_to_id, "train"
)

print("\nProcessing validation split...")
n_val, skipped_val_img, skipped_val_lbl = process_split(
    val_imgs, IMG_VAL, LBL_VAL, images_info, label_to_id, "val"
)

print(f"\n{'='*60}")
print(f"Conversion Summary:")
print(f"{'='*60}")
print(f"Train images converted: {n_train}")
print(f"  - Skipped (no image file): {skipped_train_img}")
print(f"  - Skipped (no valid labels): {skipped_train_lbl}")
print(f"\nVal images converted: {n_val}")
print(f"  - Skipped (no image file): {skipped_val_img}")
print(f"  - Skipped (no valid labels): {skipped_val_lbl}")
print(f"\nTotal: {n_train + n_val} images converted")


Processing train split...


Processing train: 100%|██████████| 24/24 [00:00<00:00, 59.99it/s]



Processing validation split...


Processing val: 100%|██████████| 6/6 [00:00<00:00, 63.60it/s]


Conversion Summary:
Train images converted: 24
  - Skipped (no image file): 0
  - Skipped (no valid labels): 0

Val images converted: 6
  - Skipped (no image file): 0
  - Skipped (no valid labels): 0

Total: 30 images converted


In [8]:
# Create data.yaml for YOLO training
data_yaml = f"""path: {YOLO_ROOT.as_posix()}

train: images/train
val: images/val

names:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space
"""

data_yaml_path = YOLO_ROOT / "data.yaml"
with open(data_yaml_path, "w") as f:
    f.write(data_yaml)

print("Created data.yaml:")
print("="*60)
print(data_yaml_path.read_text())
print("="*60)
print(f"\nFile saved to: {data_yaml_path}")


Created data.yaml:
path: c:/Harosha/George Brown/DL2/parking-vision/data/processed/yolo_parking

train: images/train
val: images/val

names:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space


File saved to: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml


## 9. Verify Dataset Structure


In [9]:
# Verify the dataset structure
print("Dataset structure verification:")
print("="*60)

# Count files
train_imgs_count = len(list(IMG_TRAIN.glob("*")))
train_lbls_count = len(list(LBL_TRAIN.glob("*.txt")))
val_imgs_count = len(list(IMG_VAL.glob("*")))
val_lbls_count = len(list(LBL_VAL.glob("*.txt")))

print(f"Train images: {train_imgs_count}")
print(f"Train labels: {train_lbls_count}")
print(f"Val images: {val_imgs_count}")
print(f"Val labels: {val_lbls_count}")

if train_imgs_count == train_lbls_count and val_imgs_count == val_lbls_count:
    print("\n✓ Dataset structure is correct (image count matches label count)")
else:
    print("\n⚠ WARNING: Image and label counts don't match!")

# Check a sample label file
if list(LBL_TRAIN.glob("*.txt")):
    sample_label = list(LBL_TRAIN.glob("*.txt"))[0]
    print(f"\nSample label file ({sample_label.name}):")
    print(sample_label.read_text()[:200])

print(f"\nDataset ready at: {YOLO_ROOT}")
print(f"Next step: Train YOLO model using notebook 05_yolo_train.ipynb")


Dataset structure verification:
Train images: 24
Train labels: 24
Val images: 6
Val labels: 6

✓ Dataset structure is correct (image count matches label count)

Sample label file (0.txt):
0 0.081633 0.171739 0.071933 0.319002
0 0.022462 0.823816 0.044925 0.311079
0 0.023262 0.169944 0.046525 0.318535
0 0.961463 0.162005 0.073192 0.315572
0 0.888008 0.162585 0.073200 0.315572
0 0.738712

Dataset ready at: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking
Next step: Train YOLO model using notebook 05_yolo_train.ipynb
